# 07 — SQL Feasibility Analysis

**Project P.U.L.S.E.** — Predictive Unified Life-sciences Summarization Engine

This notebook answers the foundational question every data scientist must answer before modelling: **Is the data actually ready?**

Using **DuckDB** — a zero-setup, in-process analytical SQL engine — we run 5 structured feasibility checks directly on the existing Parquet files without loading them fully into memory. This mirrors the triage workflow an Abbott GDSA analyst would perform at the start of a new data science engagement.

**Feasibility checks covered:**
1. Subgroup data availability (elderly diabetic cohort)
2. Feature completeness across NHANES cycles
3. Class balance assessment (label distribution)
4. Cross-cycle biomarker shift (pre-drift sanity check)
5. Minimum viable sample size validation

This report is the artefact you would hand to a senior data scientist or clinical statistician **before** starting any new modelling task.

In [ ]:
# Cell 1 — Setup
import duckdb
import pandas as pd

con = duckdb.connect()

# Register Parquet files as SQL tables (no loading into memory needed)
con.execute("CREATE VIEW master AS SELECT * FROM read_parquet('data/processed/master_patient_table.parquet')")
con.execute("CREATE VIEW nhanes_2015 AS SELECT * FROM read_parquet('data/processed/nhanes_2015.parquet')")
con.execute("CREATE VIEW nhanes_2021 AS SELECT * FROM read_parquet('data/processed/nhanes_2021.parquet')")

print("DuckDB connected. Views registered:")
con.execute("SHOW TABLES").df()

## Feasibility Check 1: Subgroup Data Availability

**Clinical question:** *Is there enough diabetic patient data in the 60+ age group for a subgroup model?*

Older patients (≥60 years) frequently present with atypical diabetes phenotypes and polypharmacy effects, making a dedicated subgroup model clinically valuable. However, reliable subgroup models require a minimum of **~500 labelled patients** — below this threshold, cross-validation confidence intervals widen to the point where model estimates are unreliable.

The query partitions each source dataset into two age brackets and reports the diabetes prevalence rate within each bracket, plus the percentage composition of the overall dataset. A count below 500 in the 60+ / diabetic cell should trigger a recommendation to pool datasets before modelling.

In [ ]:
# Cell 2 — Feasibility Check 1: Subgroup data availability
# "Is there enough diabetic patient data in the 60+ age group for a subgroup model?"
result = con.execute("""
    SELECT 
        source_dataset,
        CASE WHEN age >= 60 THEN '60+' ELSE 'Under 60' END AS age_group,
        COUNT(*) AS patient_count,
        ROUND(AVG(diabetes_label), 3) AS diabetes_rate,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY source_dataset), 1) AS pct_of_dataset
    FROM master
    WHERE diabetes_label IS NOT NULL
    GROUP BY source_dataset, age_group
    ORDER BY source_dataset, age_group
""").df()
print("=== Feasibility Check 1: Subgroup availability ===")
print(result)
# Interpretation: If 60+ count < 500, flag as insufficient for reliable subgroup model

## Feasibility Check 2: Feature Completeness Across NHANES Cycles

**Clinical question:** *How complete is HbA1c data across both NHANES cycles?*

Missing biomarker data is a leading cause of silent model degradation in production. The **70% completeness threshold** is a widely used industry heuristic: features below this level introduce imputation error that can exceed the information gain of including the feature at all, and they become a liability when the model is deployed against a population with different missingness patterns.

For an Abbott GDSA analyst, this check directly parallels the **CDISC completeness assessment** run before any clinical trial data lock — ensuring that key lab values (HbA1c, glucose) are present at sufficient rates before committing to a modelling architecture. Any feature below 70% completeness should be flagged as a **drift risk factor** in the Evidently monitoring report.

In [ ]:
# Cell 3 — Feasibility Check 2: Feature completeness across NHANES cycles
# "How complete is HbA1c data across both NHANES cycles?"
result = con.execute("""
    SELECT
        nhanes_cycle,
        COUNT(*) AS total_patients,
        COUNT(hba1c) AS hba1c_available,
        COUNT(glucose) AS glucose_available,
        COUNT(bmi) AS bmi_available,
        ROUND(COUNT(hba1c) * 100.0 / COUNT(*), 1) AS hba1c_completeness_pct,
        ROUND(COUNT(glucose) * 100.0 / COUNT(*), 1) AS glucose_completeness_pct,
        ROUND(COUNT(bmi) * 100.0 / COUNT(*), 1) AS bmi_completeness_pct
    FROM master
    WHERE source_dataset = 'nhanes'
    GROUP BY nhanes_cycle
""").df()
print("=== Feasibility Check 2: Feature completeness by NHANES cycle ===")
print(result)
# Interpretation: Highlight any feature dropping below 70% completeness as a drift risk factor

## Feasibility Check 3: Class Balance Assessment

**Clinical question:** *Is the diabetes label balanced enough to train without resampling?*

Severe class imbalance (minority class <20%) causes standard XGBoost to optimise toward accuracy on the majority class, producing **high overall accuracy but critically low recall on positive cases** — which in a clinical setting means missing real diabetic patients. The recommended responses are:

- **SMOTE** (Synthetic Minority Over-sampling Technique) — generate synthetic minority-class samples
- **`scale_pos_weight`** in XGBoost — equivalent to class weighting, simpler to implement

This check must be run against each source dataset independently, because pooling datasets with different imbalance profiles can mask the problem and produce misleading aggregate statistics.

In [ ]:
# Cell 4 — Feasibility Check 3: Class balance assessment
# "Is the diabetes label balanced enough to train without resampling?"
result = con.execute("""
    SELECT
        source_dataset,
        CAST(diabetes_label AS INTEGER) AS label,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY source_dataset), 1) AS pct
    FROM master
    WHERE diabetes_label IS NOT NULL
    GROUP BY source_dataset, diabetes_label
    ORDER BY source_dataset, label
""").df()
print("=== Feasibility Check 3: Label class balance ===")
print(result)
# Interpretation: If minority class < 20%, recommend SMOTE or class_weight='balanced'

## Feasibility Check 4: Cross-Cycle Biomarker Shift

**Clinical question:** *Have glucose and BMI distributions shifted meaningfully between NHANES cycles?*

This is a lightweight **pre-check for drift** — a statistical sanity test that precedes the full Evidently AI drift report (Layer 3 of P.U.L.S.E.). A >10% shift in mean glucose or BMI between the 2015-16 and 2021-23 cycles is strong evidence that the drift signal is real and not attributable to sampling noise.

In a **Real-World Evidence (RWE)** context — Abbott's core use case — distribution shifts like these often reflect genuine population-level health trends (e.g., increased obesity rates post-pandemic, revised clinical testing thresholds). Identifying these shifts informs both the model retraining schedule and the clinical interpretation of any downstream predictions.

**Decision rule:** If |Δmean| > 10% of the baseline value for any biomarker, escalate to a full Evidently drift analysis and flag for model retraining.

In [ ]:
# Cell 5 — Feasibility Check 4: Cross-cycle biomarker shift (pre-check for drift)
# "Have glucose and BMI distributions shifted meaningfully between NHANES cycles?"
result = con.execute("""
    SELECT
        nhanes_cycle,
        ROUND(AVG(glucose), 2)    AS avg_glucose,
        ROUND(AVG(bmi), 2)        AS avg_bmi,
        ROUND(AVG(hba1c), 2)      AS avg_hba1c,
        ROUND(STDDEV(glucose), 2) AS std_glucose,
        ROUND(STDDEV(bmi), 2)     AS std_bmi,
        ROUND(STDDEV(hba1c), 2)   AS std_hba1c
    FROM master
    WHERE source_dataset = 'nhanes'
    GROUP BY nhanes_cycle
""").df()
print("=== Feasibility Check 4: Biomarker distribution shift ===")
print(result)
# Interpretation: A >10% shift in mean glucose or BMI confirms drift is real, not sampling noise

## Feasibility Check 5: Sample Size Validation

**Clinical question:** *Do we have minimum viable sample sizes for a stratified train/test split?*

Before committing computational resources to training, this check validates that each source dataset clears the minimum sample size thresholds for the intended modelling approach:

| Labelled samples | Recommended approach |
|---|---|
| ≥ 1,000 | Full stratified 80/20 train-test split |
| 300–999 | Cross-validation only (no held-out test set) |
| < 300 | Do not model independently; pool with another source |

These thresholds are aligned with FDA guidance on minimum sample sizes for algorithmic validation in clinical decision support tools. The `cardio_feasibility` column is particularly important: the Statlog Heart dataset is small (~270 rows), which is why the cardiovascular model uses bootstrap confidence intervals rather than a standard test split.

In [ ]:
# Cell 6 — Feasibility Check 5: Sample size validation before modeling
# "Do we have minimum viable sample sizes for stratified train/test split?"
result = con.execute("""
    SELECT
        source_dataset,
        COUNT(*) AS total,
        COUNT(diabetes_label) AS labeled_diabetes,
        COUNT(cardio_label) AS labeled_cardio,
        CASE 
            WHEN COUNT(diabetes_label) >= 1000 THEN 'Sufficient for modeling'
            WHEN COUNT(diabetes_label) >= 300  THEN 'Marginal — use cross-validation only'
            ELSE 'Insufficient — do not model'
        END AS diabetes_feasibility,
        CASE 
            WHEN COUNT(cardio_label) >= 1000 THEN 'Sufficient for modeling'
            WHEN COUNT(cardio_label) >= 300  THEN 'Marginal — use cross-validation only'
            ELSE 'Insufficient — do not model'
        END AS cardio_feasibility
    FROM master
    GROUP BY source_dataset
    ORDER BY total DESC
""").df()
print("=== Feasibility Check 5: Sample size validation ===")
print(result)

## Export: Feasibility Summary CSV

The final cell produces a single-row summary table capturing the headline statistics for the entire P.U.L.S.E. dataset. This CSV is the artefact you would attach to an internal research brief or present in a data governance review meeting.

It answers the five questions a senior data scientist or clinical programme director would ask at a project kick-off:
- **Total reach** — how many patients?
- **Label coverage** — what fraction have a diabetes label?
- **Key biomarker availability** — is HbA1c present at scale?
- **Population profile** — average age and BMI

All in a single exportable row.

In [ ]:
# Cell 7 — Export feasibility summary as CSV for reporting
summary = con.execute("""
    SELECT 
        'Project P.U.L.S.E.' AS project,
        COUNT(*) AS total_patients,
        COUNT(DISTINCT source_dataset) AS datasets,
        ROUND(COUNT(diabetes_label) * 100.0 / COUNT(*), 1) AS diabetes_label_coverage_pct,
        ROUND(COUNT(hba1c) * 100.0 / COUNT(*), 1) AS hba1c_coverage_pct,
        ROUND(AVG(age), 1) AS avg_age,
        ROUND(AVG(bmi), 1) AS avg_bmi
    FROM master
""").df()
summary.to_csv("reports/feasibility_summary.csv", index=False)
print("Feasibility summary exported to reports/feasibility_summary.csv")
print(summary)